In [1]:
import json
import pandas as pd

In [3]:
from openai import OpenAI
from time import sleep


with open('openai-api-key.txt', 'r') as f:
    openai_api_key = f.read().strip()

openai_client = OpenAI(api_key=openai_api_key)

In [4]:
# separate the translation from other questions
with open('../data/structured/certamen_1996_2009.json', 'r') as f:
    questions = json.load(f)

question_df = pd.DataFrame(questions)

In [5]:
ids = question_df['question_id'].value_counts().index.tolist()
id_dict = {id: 0 for id in ids}

new_ids = []
for id_ in question_df['question_id']:
    new_id = id_ + '_' + str(id_dict[id_])
    new_ids.append(new_id)
    id_dict[id_] += 1

question_df['question_id'] = new_ids

In [6]:
# filtering:
# only questions where answers list is len 1 

filtered_df = question_df[question_df['answers'].apply(len) == 1]

# answers is single word (no whitespace)
filtered_df = filtered_df[filtered_df['answers'].apply(lambda x: ' ' not in x[0])]

# no reading comp questions
filtered_df = filtered_df[filtered_df['question_content'].apply(lambda x: 'reading' not in x.lower())]

In [2]:
translation_df = question_df[question_df['question_content'] == 'translation']
translation_df

NameError: name 'question_df' is not defined

short answer:
re-categorize question_content. Only 1 label allowed.
check question and answer language
filter out unanswerable (not enough info)

add scansion as a "question_content" option

In [8]:
short_answer = filtered_df.to_dict(orient='records')
short_answer

[{'source_name': 'NJCL-Certamen',
  'source_year': 1996,
  'question_id': 'NJCL-Certamen_1996_1a_0',
  'question_format': 'short_answer',
  'question_content': 'history',
  'difficulty': 'unknown',
  'question_language': 'english',
  'answer_language': 'latin',
  'question': 'What name was given to the large agricultural estates which resulted from the distribution of the ager publicus in the 2nd century B.C.?',
  'multiple_choice_options': [],
  'answers': ['LATIFUNDIA']},
 {'source_name': 'NJCL-Certamen',
  'source_year': 1996,
  'question_id': 'NJCL-Certamen_1996_1b_0',
  'question_format': 'short_answer',
  'question_content': 'history',
  'difficulty': 'unknown',
  'question_language': 'english',
  'answer_language': 'latin',
  'question': 'What was the manager or overseer of a latifundia called?',
  'multiple_choice_options': [],
  'answers': ['VILICUS']},
 {'source_name': 'NJCL-Certamen',
  'source_year': 1996,
  'question_id': 'NJCL-Certamen_1996_4a_0',
  'question_format': 'sh

In [9]:
len(short_answer)

5201

In [36]:
filtered_df.to_json('../data/structured/certamen_short_answer_unfiltered.json', orient='records', lines=True)

In [37]:
translation_df.to_json('../data/structured/certamen_translation_unfiltered.json', orient='records', lines=True)

1: is question answerable? can context help?

In [19]:
answerable_prompt='''I will give you a question and its answer.
Please decide if the question is answerable based on the given answer, and the added context that the question is related to Roman and Greek history, mythology, and geography, and Latin language and grammar.
Unanswerable questions don't have enough information. For example, if the question references a diagram, map, or table that is not provided, the question is unanswerable.

Please respond simply with "answerable" or "unanswerable".
'''

In [44]:
i = 50

question = short_answer[i]['question']
answer = short_answer[i]['answers'][0]
multiple_choice = short_answer[i]['multiple_choice_options']

question = 'In this diagram of a Greek theater, what is the term for the area indicated by the number 1?'
answer = 'orchestra'
multiple_choice = []

prompt = answerable_prompt 
prompt += f"\nHere is the question: {question}\n"
if multiple_choice:
    choice_str = '\n'.join(multiple_choice)
    prompt += f"Here are the multiple choice options: {choice_str}\n"
prompt += f"Answer: {answer}\n"
print(prompt)

I will give you a question and its answer.
Please decide if the question is answerable based on the given answer, and the added context that the question is related to Roman and Greek history, mythology, and geography, and Latin language and grammar.
Unanswerable questions don't have enough information. For example, if the question references a diagram, map, or table that is not provided, the question is unanswerable.

Please respond simply with "answerable" or "unanswerable".

Here is the question: In this diagram of a Greek theater, what is the term for the area indicated by the number 1?
Answer: orchestra



In [15]:
response = openai_client.responses.create(
    model="gpt-4o",
    input=prompt
)
resp = response.output[0].content[0].text

In [45]:
response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    )
resp = response.choices[0].message.content

In [46]:
resp

'unanswerable'

In [42]:
# load previous responses
with open('../data/structured/certamen_answerable_responses.json', 'r') as f:
    responses = json.load(f)


In [43]:
len(responses)

1501

In [45]:
len(short_answer)

5201

In [48]:
responses = []

In [ ]:
N = len(short_answer)

# continue from where we left off, or from first error response
for i in range(len(short_answer)):
    #if i < len(responses) and responses[i] != 'error':
    #    continue
    print(i, end=' ')

    q_id = short_answer[i]['question_id']
    question = short_answer[i]['question']
    answer = short_answer[i]['answers'][0]
    multiple_choice = short_answer[i]['multiple_choice_options']

    prompt = answerable_prompt 
    prompt += f"\nHere is the question: {question}\n"
    if multiple_choice:
        choice_str = '\n'.join(multiple_choice)
        prompt += f"Here are the multiple choice options: {choice_str}\n"
    prompt += f"Answer: {answer}\n"
    
    try:
        response = openai_client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": prompt}
                ]
            )
        resp = response.choices[0].message.content
    except Exception as e:
        print(f"Error on question {i}: {e}")
        resp = 'error'
    
    #if responses[i] == 'error':
    #    responses[i] = resp
    #elif i < len(responses):
    responses.append(resp)

    if i % 50 == 0:
        print(f"{i+1}/{N}")
        # dump to json
        with open('../data/structured/certamen_answerable_responses.json', 'w') as f:
            json.dump(responses, f, indent=4)

    sleep(.01)
    




In [50]:
# dump final responses
with open('../data/structured/certamen_answerable_responses.json', 'w') as f:
    json.dump(responses, f, indent=4)


In [51]:
len(responses)

5201

In [54]:
from collections import Counter
lower_resp = [r.lower() for r in responses]
lower_resp = [r.replace('.', '') for r in lower_resp]
count_resp = Counter(lower_resp)
count_resp




Counter({'answerable': 4643, 'unanswerable': 558})

In [55]:
for i in range(len(short_answer)):
    short_answer[i]['answerable'] = responses[i]

short_answer_df = pd.DataFrame(short_answer)
short_answer_df[short_answer_df['answerable'] == 'unanswerable']

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,answerable
4,NJCL-Certamen,1996,NJCL-Certamen_1996_7a_0,short_answer,grammar,unknown,english,latin,"Give the first person singular, present passiv...",[],[VEHAR],unanswerable
5,NJCL-Certamen,1996,NJCL-Certamen_1996_7b_0,short_answer,grammar,unknown,english,latin,Change vehar to imperfect.,[],[VEHERER],unanswerable
17,NJCL-Certamen,1996,NJCL-Certamen_1996_16c_0,short_answer,translation,unknown,english,latin,Translate into Latin the relative pronoun for ...,[],[CUĪ],unanswerable
35,NJCL-Certamen,1996,NJCL-Certamen_1996_7e_1,short_answer,history,unknown,english,latin,To what Germanic tribe did Odo(v)acer belong?,[],[OSTROGOTHS],unanswerable
36,NJCL-Certamen,1996,NJCL-Certamen_1996_7f_1,short_answer,history,unknown,english,latin,Who was the emperor in the eastern empire at t...,[],[ZENO],unanswerable
...,...,...,...,...,...,...,...,...,...,...,...,...
5167,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_877,short_answer,vocabulary,advanced,english,latin,When this consultant reads an email from his o...,[],[RĒ],unanswerable
5173,NJCL-Certamen,2002,NJCL-Certamen_2002_10_71,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[sojourn, conjure, meridian, circadian, diary]",[CONJURE],unanswerable
5174,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_870,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...","[vintner, vinyl, vicious, vignette, vinegar]",[VICIOUS],unanswerable
5183,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_878,short_answer,literature,unknown,english,latin,Who had issued this decree?,[],[CREON],unanswerable


In [56]:
unanswerable_df = short_answer_df[short_answer_df['answerable'] == 'unanswerable']


In [57]:
unanswerable_df['question_content'].value_counts()

question_content
vocabulary          143
mythology           109
grammar              86
history              84
geography            40
literature           37
literary_devices     14
translation           7
Name: count, dtype: int64

In [58]:
unanswerable_df[unanswerable_df['question_content'] == 'translation']

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,answerable
17,NJCL-Certamen,1996,NJCL-Certamen_1996_16c_0,short_answer,translation,unknown,english,latin,Translate into Latin the relative pronoun for ...,[],[CUĪ],unanswerable
41,NJCL-Certamen,1996,NJCL-Certamen_1996_10c_1,short_answer,translation,unknown,english,latin,"Using the verb volō, velle translate “want” in...",[],[VELLE],unanswerable
103,NJCL-Certamen,1996,NJCL-Certamen_1996_19c_3,short_answer,translation,unknown,english,latin,"Using cēnō, cēnāre, translate into Latin the v...",[],[CĒNĀRE],unanswerable
237,NJCL-Certamen,1996,NJCL-Certamen_1996_6c_6,short_answer,translation,unknown,english,latin,Translate “yourself’ in this sentence: Quintus...,[],[TIBI],unanswerable
1012,NJCL-Certamen,1996,NJCL-Certamen_1996_8a_31,short_answer,translation,unknown,english,latin,Translate into Latin the word “fly” for the se...,[],[VOLĀVISSE],unanswerable
2142,NJCL-Certamen,2000,NJCL-Certamen_2000_20a_8,short_answer,translation,unknown,english,english,How should one translate the verb form amābō i...,[],[PLEASE],unanswerable
4050,NJCL-Certamen,2002,NJCL-Certamen_2002_13b_50,short_answer,translation,advanced,english,english,The seal also contains the year in which the u...,[],[1820],unanswerable


In [59]:
unanswerable_df[unanswerable_df['question_content'] == 'grammar']

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,answerable
4,NJCL-Certamen,1996,NJCL-Certamen_1996_7a_0,short_answer,grammar,unknown,english,latin,"Give the first person singular, present passiv...",[],[VEHAR],unanswerable
5,NJCL-Certamen,1996,NJCL-Certamen_1996_7b_0,short_answer,grammar,unknown,english,latin,Change vehar to imperfect.,[],[VEHERER],unanswerable
53,NJCL-Certamen,1996,NJCL-Certamen_1996_20b_0,short_answer,grammar,unknown,english,latin,"Give the present active participle of dlrigō, ...",[],[dīrigēns],unanswerable
307,NJCL-Certamen,1996,NJCL-Certamen_1996_19a_8,short_answer,grammar,unknown,english,latin,Give the comparative form of levis.,[],[LEVISSIMUS],unanswerable
308,NJCL-Certamen,1996,NJCL-Certamen_1996_19b_8,short_answer,grammar,unknown,english,latin,Give the superlative form of levis.,[],[LEVIOR],unanswerable
...,...,...,...,...,...,...,...,...,...,...,...,...
4983,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_774,short_answer,grammar,unknown,english,english,You are looking at an inscription that was ori...,[],[Cause],unanswerable
5011,NJCL-Certamen,2002,NJCL-Certamen_2002_4b_50,short_answer,grammar,unknown,english,latin,Make that form fuerim imperfect.,[],[ESSEM],unanswerable
5054,NJCL-Certamen,2002,NJCL-Certamen_2002_18_73,short_answer,grammar,unknown,english,latin,"For the verb gerō, give the singular form of t...",[],[GERERE],unanswerable
5067,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_822,short_answer,grammar,advanced,english,latin,Give the correct form of the adjective prūdēns...,[],[PRŪDENTIA],unanswerable


In [61]:
# dump unanswerable questions to json
with open('../data/structured/certamen_unanswerable_questions.tsv', 'w') as f:
    unanswerable_df.to_csv(f, sep='\t')
#unanswerable_df.to_json('../data/structured/certamen_unanswerable_questions.json', orient='records', lines=True)


In [63]:
# load edited unanswerable questions
with open('../data/structured/certamen_unanswerable_edits.tsv', 'r') as f:
    unanswerable_df = pd.read_csv(f, sep='\t')

# drop col "unnamed: 0"
unanswerable_df = unanswerable_df.drop(columns=['Unnamed: 0'])
unanswerable_df




,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers,answerable,edits
0,NJCL-Certamen,1996,NJCL-Certamen_1996_7a_0,short_answer,grammar,unknown,english,latin,"Give the first person singular, present passiv...",[],['VEHAR'],1,NaN
1,NJCL-Certamen,1996,NJCL-Certamen_1996_7b_0,short_answer,grammar,unknown,english,latin,Change vehar to imperfect.,[],['VEHERER'],1,NaN
2,NJCL-Certamen,1996,NJCL-Certamen_1996_16c_0,short_answer,translation,unknown,english,latin,Translate into Latin the relative pronoun for ...,[],['CUĪ'],1,NaN
3,NJCL-Certamen,1996,NJCL-Certamen_1996_7e_1,short_answer,history,unknown,english,latin,To what Germanic tribe did Odo(v)acer belong?,[],['OSTROGOTHS'],0,NaN
4,NJCL-Certamen,1996,NJCL-Certamen_1996_7f_1,short_answer,history,unknown,english,latin,Who was the emperor in the eastern empire at t...,[],['ZENO'],0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
515,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_877,short_answer,vocabulary,advanced,english,latin,When this consultant reads an email from his o...,[],['RĒ'],1,NaN
516,NJCL-Certamen,2002,NJCL-Certamen_2002_10_71,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...",[],['CONJURE'],1,1.0
517,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_870,multiple_choice,vocabulary,unknown,english,english,"Which of the following, if any, is NOT derived...",[],['VICIOUS'],1,1.0
518,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_878,short_answer,literature,unknown,english,latin,Who had issued this decree?,[],['CREON'],0,NaN


In [64]:
# what q ids are unanswerable?
# ids where answerable = 0
unanswerable_ids = unanswerable_df[unanswerable_df['answerable'] == 0]['question_id'].tolist()
unanswerable_ids


['NJCL-Certamen_1996_7e_1',
 'NJCL-Certamen_1996_7f_1',
 'NJCL-Certamen_1996_20b_0',
 'NJCL-Certamen_1996_2a_4',
 'NJCL-Certamen_1996_2b_4',
 'NJCL-Certamen_1996_17c_4',
 'NJCL-Certamen_1996_20c_4',
 'NJCL-Certamen_1996_1a_8',
 'NJCL-Certamen_1996_1b_8',
 'NJCL-Certamen_1996_20c_10',
 'NJCL-Certamen_1996_7c_12',
 'NJCL-Certamen_1996_19b_11',
 'NJCL-Certamen_1996_6c_11',
 'NJCL-Certamen_1996_19b_13',
 'NJCL-Certamen_1996_1a_20',
 'NJCL-Certamen_1996_1b_20',
 'NJCL-Certamen_1996_6c_15',
 'NJCL-Certamen_1996_17a_17',
 'NJCL-Certamen_1996_11_2',
 'NJCL-Certamen_1996_12_2',
 'NJCL-Certamen_1996_12c_17',
 'NJCL-Certamen_1996_14b_19',
 'NJCL-Certamen_1996_11c_22',
 'NJCL-Certamen_1996_10d_0',
 'NJCL-Certamen_1996_5b_33',
 'NJCL-Certamen_1996_8a_17',
 'NJCL-Certamen_1996_8c_17',
 'NJCL-Certamen_1996_1b_39',
 'NJCL-Certamen_1996_7b_31',
 'NJCL-Certamen_1996_13b_29',
 'NJCL-Certamen_1996_15b_30',
 'NJCL-Certamen_1996_15c_30',
 'NJCL-Certamen_1996_7a_42',
 'NJCL-Certamen_1996_7b_41',
 'NJCL-Certa

In [65]:
len(unanswerable_ids)

260

In [67]:
# ids where edit = 1
update_ids = unanswerable_df[unanswerable_df['edits'] == 1]['question_id'].tolist()
update_ids

30

In [68]:
len(short_answer_df)

5201

In [69]:
# drop unanswerable from short_answer_df
short_answer_df = short_answer_df[~short_answer_df['question_id'].isin(unanswerable_ids)]
len(short_answer_df)

4941

In [72]:
short_answer_dict = short_answer_df.to_dict(orient='records')
for i, dict_ in enumerate(short_answer_dict):
    this_id = dict_['question_id']
    if this_id in update_ids:
        # find the row in unanswerable_df
        new_row = unanswerable_df[unanswerable_df['question_id'] == this_id]
        # drop the "edits" column
        new_row = new_row.drop(columns=['edits'])
        # turn into dict
        new_row = new_row.to_dict(orient='records')[0]
        # update the row
        short_answer_dict[i] = new_row
    # delete the "answerable" key 
    del short_answer_dict[i]['answerable']

short_answer_dict



[{'source_name': 'NJCL-Certamen',
  'source_year': 1996,
  'question_id': 'NJCL-Certamen_1996_1a_0',
  'question_format': 'short_answer',
  'question_content': 'history',
  'difficulty': 'unknown',
  'question_language': 'english',
  'answer_language': 'latin',
  'question': 'What name was given to the large agricultural estates which resulted from the distribution of the ager publicus in the 2nd century B.C.?',
  'multiple_choice_options': [],
  'answers': ['LATIFUNDIA']},
 {'source_name': 'NJCL-Certamen',
  'source_year': 1996,
  'question_id': 'NJCL-Certamen_1996_1b_0',
  'question_format': 'short_answer',
  'question_content': 'history',
  'difficulty': 'unknown',
  'question_language': 'english',
  'answer_language': 'latin',
  'question': 'What was the manager or overseer of a latifundia called?',
  'multiple_choice_options': [],
  'answers': ['VILICUS']},
 {'source_name': 'NJCL-Certamen',
  'source_year': 1996,
  'question_id': 'NJCL-Certamen_1996_4a_0',
  'question_format': 'sh

In [73]:
len(short_answer_dict)

4941

In [74]:
with open('../data/structured/certamen_short_answer_filtered_answerable.json', 'w') as f:
    json.dump(short_answer_dict, f, indent=4)

In [ ]:
# dump remaining responses
with open('../data/structured/certamen_answerable_responses.json', 'w') as f:
    json.dump(responses, f, indent=4)


filtering translation questions

In [6]:
# load translation questions (jsonl)
with open('../data/structured/certamen_translation_unfiltered.json', 'r') as f:
    translation_dicts = [json.loads(line) for line in f]

translation_df = pd.DataFrame(translation_dicts)
translation_df



,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers
0,NJCL-Certamen,1996,NJCL-Certamen_1996_11b_0,short_answer,translation,unknown,english,english,Translate adeō as an adverb.,[],"[SO FAR, SO MUCH, MOREOVER]"
1,NJCL-Certamen,1996,NJCL-Certamen_1996_11c_0,short_answer,translation,unknown,english,english,Translate adeō as a verb.,[],[I APPROACH]
2,NJCL-Certamen,1996,NJCL-Certamen_1996_16a_0,short_answer,translation,unknown,english,latin,Translate into Latin the relative pronoun for ...,[],[CUIUS]
3,NJCL-Certamen,1996,NJCL-Certamen_1996_16b_0,short_answer,translation,unknown,english,latin,Translate into Latin the relative pronoun for ...,[],[QUIBUS(CUM)]
4,NJCL-Certamen,1996,NJCL-Certamen_1996_16c_0,short_answer,translation,unknown,english,latin,Translate into Latin the relative pronoun for ...,[],[CUĪ]
...,...,...,...,...,...,...,...,...,...,...,...
1128,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_865,short_answer,translation,advanced,english,english,Translate the following sentence into English:...,[],[IT IS NOT CONVENIENT / COMFORTABLE / FITTING ...
1129,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_874,short_answer,translation,advanced,english,english,Translate the following sentence into English:...,[],[IT IS PROPER / BECOMING / FITTING FOR ALL / E...
1130,NJCL-Certamen,2002,NJCL-Certamen_2002_13_77,short_answer,translation,advanced,latin,english,Translate the following sentence into English:...,[],[My brother saw the dog wounded by a rock]
1131,NJCL-Certamen,2002,NJCL-Certamen_2002_13a_71,short_answer,translation,advanced,latin,english,Translate the following sentence into English:...,[],"[Having seen the wounded dog, my brother wante..."


In [9]:
# translation >= 3 words
long_translation_df = translation_df[translation_df['answers'].apply(lambda x: len(x[0].split()) >= 3)]
long_translation_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers
8,NJCL-Certamen,1996,NJCL-Certamen_1996_8a_0,short_answer,translation,unknown,english,english,Translate the motto of Alabama: Audēmus iūra n...,[],"[WE DARE TO DEFEND OUR RIGHTS, WE DARE TO DEFE..."
9,NJCL-Certamen,1996,NJCL-Certamen_1996_8b_0,short_answer,translation,unknown,english,english,Translate the motto of the Dominion of Canada:...,[],[FROM SEA TO SEA]
10,NJCL-Certamen,1996,NJCL-Certamen_1996_8c_0,short_answer,translation,unknown,english,english,Translate the motto of Wellesley College: nōn ...,[],"[NOT TO BE SERVED, NOT TO BE MANAGED, BUT TO S..."
16,NJCL-Certamen,1996,NJCL-Certamen_1996_15a_3,short_answer,translation,unknown,english,english,Translate into English the following maxim of ...,[],"[If you want peace, prepare for war]"
17,NJCL-Certamen,1996,NJCL-Certamen_1996_15b_3,short_answer,translation,unknown,english,english,Translate into English the following quotation...,[],"[Divine nature gave (us) the fields, human art..."
...,...,...,...,...,...,...,...,...,...,...,...
1128,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_865,short_answer,translation,advanced,english,english,Translate the following sentence into English:...,[],[IT IS NOT CONVENIENT / COMFORTABLE / FITTING ...
1129,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_874,short_answer,translation,advanced,english,english,Translate the following sentence into English:...,[],[IT IS PROPER / BECOMING / FITTING FOR ALL / E...
1130,NJCL-Certamen,2002,NJCL-Certamen_2002_13_77,short_answer,translation,advanced,latin,english,Translate the following sentence into English:...,[],[My brother saw the dog wounded by a rock]
1131,NJCL-Certamen,2002,NJCL-Certamen_2002_13a_71,short_answer,translation,advanced,latin,english,Translate the following sentence into English:...,[],"[Having seen the wounded dog, my brother wante..."


In [37]:
# long translation  df with more than 1 answer
long_translation_df[long_translation_df['answers'].apply(lambda x: len(x) > 1)]

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers
8,NJCL-Certamen,1996,NJCL-Certamen_1996_8a_0,short_answer,translation,unknown,english,english,Translate the motto of Alabama: Audēmus iūra n...,[],"[WE DARE TO DEFEND OUR RIGHTS, WE DARE TO DEFE..."
10,NJCL-Certamen,1996,NJCL-Certamen_1996_8c_0,short_answer,translation,unknown,english,english,Translate the motto of Wellesley College: nōn ...,[],"[NOT TO BE SERVED, NOT TO BE MANAGED, BUT TO S..."
18,NJCL-Certamen,1996,NJCL-Certamen_1996_15c_3,short_answer,translation,unknown,english,english,Translate into English the following quotation...,[],"[What an outstanding guard of sheep, the wolf,..."
19,NJCL-Certamen,1996,NJCL-Certamen_1996_16_0,short_answer,translation,unknown,english,english,Translate the Caesarean idiom dolōrem capere.,[],"[To be grieved, Suffer grief, Grieve]"
25,NJCL-Certamen,1996,NJCL-Certamen_1996_6c_3,short_answer,translation,unknown,english,english,Translate this adage into English: Nonne habēs...,[],[SURELY YOU HAVE SOMETHING (WHICH) YOU WANT TO...
...,...,...,...,...,...,...,...,...,...,...,...
1109,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_834,short_answer,translation,unknown,english,english,Translate the following sentence into English:...,[],"[JULIA IS THE PRETTY DAUGHTER OF THE DICTATOR,..."
1110,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_843,short_answer,translation,unknown,english,english,Translate the following sentence into English:...,[],"[LUCIUS WAS A VERY FAMOUS COOK, LUCIUS WAS THE..."
1119,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_848,short_answer,translation,advanced,english,english,Translate the Latin motto for St. Mary’s Colle...,[],"[STANDARD OF FAITH, SIGN OF FAITH, SIGNAL OF F..."
1121,NJCL-Certamen,2002,NJCL-Certamen_2002_17_79,short_answer,translation,unknown,english,english,Translate the following sentence into English:...,[],[SURELY THE SOLDIERS ARE MORE BRAVE THAN GLADI...


In [11]:
# which answers have multiple possible answers? we should add them as alternative translations
# look for answers that have / or () in them

multiple_refs_df = long_translation_df[long_translation_df['answers'].apply(lambda x: '/' in x[0] or '(' in x[0])]
multiple_refs_df

,source_name,source_year,question_id,question_format,question_content,difficulty,question_language,answer_language,question,multiple_choice_options,answers
17,NJCL-Certamen,1996,NJCL-Certamen_1996_15b_3,short_answer,translation,unknown,english,english,Translate into English the following quotation...,[],"[Divine nature gave (us) the fields, human art..."
25,NJCL-Certamen,1996,NJCL-Certamen_1996_6c_3,short_answer,translation,unknown,english,english,Translate this adage into English: Nonne habēs...,[],[SURELY YOU HAVE SOMETHING (WHICH) YOU WANT TO...
30,NJCL-Certamen,1996,NJCL-Certamen_1996_20b_3,short_answer,translation,unknown,latin,english,Dīc mihi Anglicē cur certāmen amēs.,[],[PLAYER SHOULD TELL YOU WHY HE (SHE) LOVES CER...
31,NJCL-Certamen,1996,NJCL-Certamen_1996_20c_3,short_answer,translation,unknown,latin,english,Dīcite mihi Anglicē unde vēnerītis.,[],[ALL PLAYERS SHOULD SAY WHERE (WHAT STATE) THE...
38,NJCL-Certamen,1996,NJCL-Certamen_1996_16a_6,short_answer,translation,unknown,english,english,Translate: Primā lūce servi ē villā in agrōs ī...,[],"[AT DAWN, THE SLAVES WENT OUT OF THE HOUSE (AN..."
...,...,...,...,...,...,...,...,...,...,...,...
1127,NJCL-Certamen,2002,NJCL-Certamen_2002_3_78,short_answer,translation,advanced,english,english,Translate the following sentence into English:...,[],[IT IS NECSESARY FOR YOU TO / YOU MUST LISTEN ...
1128,NJCL-Certamen,2002,NJCL-Certamen_2002_B1_865,short_answer,translation,advanced,english,english,Translate the following sentence into English:...,[],[IT IS NOT CONVENIENT / COMFORTABLE / FITTING ...
1129,NJCL-Certamen,2002,NJCL-Certamen_2002_B2_874,short_answer,translation,advanced,english,english,Translate the following sentence into English:...,[],[IT IS PROPER / BECOMING / FITTING FOR ALL / E...
1131,NJCL-Certamen,2002,NJCL-Certamen_2002_13a_71,short_answer,translation,advanced,latin,english,Translate the following sentence into English:...,[],"[Having seen the wounded dog, my brother wante..."


In [12]:
multiple_refs_dict = multiple_refs_df.to_dict(orient="records")
multiple_refs_dict


[{'source_name': 'NJCL-Certamen',
  'source_year': 1996,
  'question_id': 'NJCL-Certamen_1996_15b_3',
  'question_format': 'short_answer',
  'question_content': 'translation',
  'difficulty': 'unknown',
  'question_language': 'english',
  'answer_language': 'english',
  'question': 'Translate into English the following quotation from Tibullus: Dīvīna nātūra dedit agrōs, ars hūmāna aedificāvit urbēs.',
  'multiple_choice_options': [],
  'answers': ['Divine nature gave (us) the fields, human art built the cities']},
 {'source_name': 'NJCL-Certamen',
  'source_year': 1996,
  'question_id': 'NJCL-Certamen_1996_6c_3',
  'question_format': 'short_answer',
  'question_content': 'translation',
  'difficulty': 'unknown',
  'question_language': 'english',
  'answer_language': 'english',
  'question': 'Translate this adage into English: Nonne habēs aliquid quod mihi monstrāre vīs?',
  'multiple_choice_options': [],
  'answers': ['SURELY YOU HAVE SOMETHING (WHICH) YOU WANT TO SAY TO SHOW ME',
   '

In [16]:
multiple_refs_prompt = '''I will give you the answer key for a translation question. The key denotes multiple possible translations that are correct by enclosing alternatives in parentheses or using a slash. 
Please expand the key into a list of all possible translations, where each translation stands alone without parentheses or a slash.
Do not add any new translations; only expand the answer key given.
Please respond ONLY with the expanded list of translations, each on a new line. Do not include any other text.
'''

latin_ans_prompt = '''The translation is in Latin, so for reference I will also give you the original question in English.'''

In [23]:
i = 101

question = multiple_refs_dict[i]['question']
answers = multiple_refs_dict[i]['answers']
answer_language = multiple_refs_dict[i]['answer_language']


prompt = multiple_refs_prompt
if answer_language == 'latin':
    prompt += f"\n{latin_ans_prompt}\nOriginal English question: {question}"

prompt += "\nPossible translations:\n" + '\n'.join(answers)

print(prompt)


I will give you the answer key for a translation question. The key denotes multiple possible translations that are correct by enclosing alternatives in parentheses or using a slash. 
Please expand the key into a list of all possible translations, where each translation stands alone without parentheses or a slash.
Do not add any new translations; only expand the answer key given.
Please respond ONLY with the expanded list of translations, each on a new line. Do not include any other text.

The translation is in Latin, so for reference I will also give you the original question in English.
Original English question: Using the dative, say in Latin “This dog is a great help to me.”
Possible translations:
IS / EA or HIC / HAEC CANIS MIHI EST MAGNŌ AUXILIŌ / SUBSIDIŌ / ŪSUĪ
MAGNAE OPĪ


In [24]:
response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt}
        ]
    )
resp = response.choices[0].message.content
resp

'IS CANIS MIHI EST MAGNŌ AUXILIŌ  \nIS CANIS MIHI EST MAGNŌ SUBSIDIŌ  \nIS CANIS MIHI EST MAGNŌ ŪSUĪ  \nIS CANIS MIHI EST MAGNAE OPĪ  \nEA CANIS MIHI EST MAGNŌ AUXILIŌ  \nEA CANIS MIHI EST MAGNŌ SUBSIDIŌ  \nEA CANIS MIHI EST MAGNŌ ŪSUĪ  \nEA CANIS MIHI EST MAGNAE OPĪ  \nHIC CANIS MIHI EST MAGNŌ AUXILIŌ  \nHIC CANIS MIHI EST MAGNŌ SUBSIDIŌ  \nHIC CANIS MIHI EST MAGNŌ ŪSUĪ  \nHIC CANIS MIHI EST MAGNAE OPĪ  \nHAEC CANIS MIHI EST MAGNŌ AUXILIŌ  \nHAEC CANIS MIHI EST MAGNŌ SUBSIDIŌ  \nHAEC CANIS MIHI EST MAGNŌ ŪSUĪ  \nHAEC CANIS MIHI EST MAGNAE OPĪ  '

In [26]:
responses = []
N = len(multiple_refs_dict)
for i in range(len(multiple_refs_dict)):
    #if i < len(responses) and responses[i] != 'error':
    #    continue
    print(i, end=' ')

    q_id = multiple_refs_dict[i]['question_id']
    question = multiple_refs_dict[i]['question']
    answers = multiple_refs_dict[i]['answers']
    answer_language = multiple_refs_dict[i]['answer_language']

    prompt = multiple_refs_prompt
    if answer_language == 'latin':
        prompt += f"\n{latin_ans_prompt}\nOriginal English question: {question}"
    prompt += "\nPossible translations:\n" + '\n'.join(answers)

    
    try:
        response = openai_client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": "You are a helpful assistant."},
                    {"role": "user", "content": prompt}
                ]
            )
        resp = response.choices[0].message.content
    except Exception as e:
        print(f"Error on question {i}: {e}")
        resp = 'error'
    
    #if responses[i] == 'error':
    #    responses[i] = resp
    #elif i < len(responses):
    responses.append(resp)

    if i % 50 == 0:
        print(f"{i+1}/{N}")
        # dump to json
        with open('../data/structured/certamen_translation_expanded_responses.json', 'w') as f:
            json.dump(responses, f, indent=4)

    sleep(.01)

# save final responses
with open('../data/structured/certamen_translation_expanded_responses.json', 'w') as f:
    json.dump(responses, f, indent=4)


0 1/405
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51/405
51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 73 74 75 76 77 78 79 80 81 82 83 84 85 86 87 88 89 90 91 92 93 94 95 96 97 98 99 100 101/405
101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 151/405
151 152 153 154 155 156 157 158 159 160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197 198 199 200 201/405
201 202 203 204 205 206 207 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 239 240 241 242 243 244 245 246 247 248 249 250 251/405
251 252 253 254 255 256 257 258 259 260 261 262 263 264 265 2

In [27]:
len(responses), len(multiple_refs_dict)

(405, 405)

In [32]:
# parse responses
new_responses = []
for resp in responses:
    new_resp = resp.split('\n')
    
    # clean up whitespace
    new_resp = [r.strip() for r in new_resp]
    
    # remove duplicates 
    new_resp = list(set(new_resp))
    
    # remove empty strings
    new_resp = [r for r in new_resp if r]

    assert new_resp != []
    
    new_responses.append(new_resp)


In [35]:
#avg answer len

total_len = sum([len(r) for r in new_responses])
max_len = max([len(r) for r in new_responses])
min_len = min([len(r) for r in new_responses])
avg_len = total_len / len(new_responses)
print(f"Average number of translations per answer: {avg_len}")
print(f"Max number of translations per answer: {max_len}")
print(f"Min number of translations per answer: {min_len}")


Average number of translations per answer: 4.241975308641975
Max number of translations per answer: 48
Min number of translations per answer: 1


In [39]:
# add back to multiple_refs_dict
for i, dict_ in enumerate(multiple_refs_dict):
    dict_['answers'] = new_responses[i]

# update multiple_refs_df
multiple_refs_df = pd.DataFrame(multiple_refs_dict)

In [38]:
# multiple_refs_ids
multiple_refs_ids = multiple_refs_df['question_id'].tolist()

In [41]:
# update long_translation_df with the new multiple_refs answers 
long_translation_dict = long_translation_df.to_dict(orient="records")
for i, old_dict in enumerate(long_translation_dict):
    if old_dict['question_id'] in multiple_refs_ids:
        # find the row in multiple_refs_df
        new_dict = multiple_refs_df[multiple_refs_df['question_id'] == old_dict['question_id']].to_dict(orient="records")[0]
        # update the row
        long_translation_dict[i] = new_dict




In [43]:
len(long_translation_dict)

960

In [44]:
# dump json
with open('../data/structured/certamen_translation_long.json', 'w') as f:
    json.dump(long_translation_dict, f, indent=4)



